# Mono-Camera Example — JAX Accelerated

This notebook is a JAX-accelerated version of `mono_camera_example.ipynb`.
The physics, simulation, and downstream analysis are identical — only the
observability matrix computation is replaced.

## What changes vs. the original notebook

| Step | Original | JAX version |
|------|----------|-------------|
| Simulation | `Simulator` (do_mpc / CasADi IDAS) | **unchanged** |
| Observability matrix | `SlidingEmpiricalObservabilityMatrix` — 2n perturbation sims per window | `JaxSlidingEmpiricalObservabilityMatrix` — one `jax.vmap(jax.jacfwd)` call for all windows |
| Fisher information | `SlidingFisherObservability` | **unchanged** |
| Plotting | same | **unchanged** |

### Typical speedup
On 895 sliding windows (n=2 states, w=6 time-steps):
* Legacy numerical: ~21 s
* JAX vmap (after JIT warmup): ~1 s → **~19× faster**

### What you need to do differently
1. Install JAX: `pip install "jax[cpu]"`
2. Write a JAX-compatible version of your dynamics `f_jax` and measurement `h_jax`
   using `jax.numpy` instead of plain `numpy` for any math operations.
3. Create a `JaxSimulator` with the JAX functions.
4. Replace `SlidingEmpiricalObservabilityMatrix(simulator, ...)` with
   `JaxSlidingEmpiricalObservabilityMatrix(jax_simulator, ...)`.

Everything else is identical to the original notebook.

# Install pybounds and requirements

This should allow the notebook to run directly in Google Colab.

In [ ]:
try:
    import casadi
except:
    !pip install casadi
    import casadi

try:
    import do_mpc
except:
    !pip install do_mpc
    import do_mpc

try:
    import pybounds
except:
    !pip install pybounds
    import pybounds

try:
    import jax
except:
    !pip install "jax[cpu]"
    import jax

# Import packages

In [ ]:
import time
import numpy as np
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt

from pybounds import (Simulator,
                      JaxSimulator, JaxSlidingEmpiricalObservabilityMatrix,
                      FisherObservability, SlidingFisherObservability,
                      ObservabilityMatrixImage, colorline)

# Define system dynamics and measurements

This example uses a simple 2-state linear system with a single nonlinear
measurement — a downward-pointed camera moving horizontally.

States: ground speed $g$, distance above the ground $d$

$$
\dot{\mathbf{x}} = \begin{bmatrix} \dot{g} \\ \dot{d} \end{bmatrix}
= f(\mathbf{x}, u) = \begin{bmatrix} u \\ 0 \end{bmatrix}
$$

Measurement: ventral optic flow $r = g/d$

$$
\mathbf{y} = h(\mathbf{x}) = \begin{bmatrix} g/d \end{bmatrix}
$$

In [ ]:
state_names       = ['g', 'd']
input_names       = ['u']
measurement_names = ['r']
dt = 0.01  # [s]

## Legacy dynamics and measurement (for `Simulator` / do_mpc)

These are used by the `Simulator` to integrate the trajectory.
They work with CasADi symbolic types as well as plain Python lists.

In [ ]:
def f(X, U):
    g, d = X
    u = U[0]
    return [u, 0*u]

def h(X, U):
    g, d = X
    return [g/d]

## JAX dynamics and measurement (for `JaxSimulator`)

These are the **only new functions you need to write** compared to the original
notebook.

Rules:
* Arguments `x` and `u` are 1-D `jnp` arrays (indexed by position, not name).
* Return a `jnp.array(...)` — do **not** return a plain Python list.
* Use `jnp.*` for any math (sin, cos, sqrt, …). Plain Python arithmetic
  (`+`, `-`, `*`, `/`) works as-is.

For this simple system the only difference is the return type (`jnp.array`
instead of a list).

In [ ]:
def f_jax(x, u):
    # x[0] = g,  x[1] = d
    return jnp.array([u[0], 0.0 * u[0]])

def h_jax(x, u):
    return jnp.array([x[0] / x[1]])

# Create simulator objects

* **`Simulator`** (do_mpc / CasADi) — used to integrate the trajectory and,
  optionally, to run MPC.  **Unchanged from original notebook.**
* **`JaxSimulator`** — pure JAX forward integrator used only for the
  observability computation.

In [ ]:
# Legacy simulator — unchanged
simulator = Simulator(f, h, dt=dt,
                      state_names=state_names,
                      input_names=input_names,
                      measurement_names=measurement_names)

# JAX simulator — same parameters, JAX-compatible functions
jax_simulator = JaxSimulator(f_jax, h_jax, dt=dt,
                             state_names=state_names,
                             input_names=input_names,
                             measurement_names=measurement_names,
                             integrator='rk4')

# Simulate

Use the legacy `Simulator` to integrate the trajectory.  The output
`(t_sim, x_sim, u_sim)` is then passed to the JAX observability class.

In [ ]:
x0 = {'g': 2.0, 'd': 3.0}

tsim1 = np.arange(0, 3, step=dt)
tsim2 = np.arange(3, 6, step=dt)
tsim3 = np.arange(6, 9, step=dt)

u1 = np.sin(3 * tsim1)
u2 = 1e-5 * np.sin(3 * tsim2)   # near-zero segment → low observability
u3 = np.sin(3 * tsim3)
u  = dict(u=np.hstack((u1, u2, u3)))

t_sim, x_sim, u_sim, y_sim = simulator.simulate(x0=x0, u=u, return_full_output=True)

In [ ]:
simulator.plot('x')

# Observability

## Construct sliding observability matrix — JAX version

This is the **one line that changes** relative to the original notebook:

```python
# Original (numerical perturbations — slow)
SEOM = SlidingEmpiricalObservabilityMatrix(simulator, t_sim, x_sim, u_sim, w=w, eps=1e-4)

# JAX (exact Jacobian via vmap + jacfwd — fast)
SEOM = JaxSlidingEmpiricalObservabilityMatrix(jax_simulator, t_sim, x_sim, u_sim, w=w)
```

The output `SEOM.O_df_sliding` has the **same format** as before, so all
downstream code (`FisherObservability`, `SlidingFisherObservability`, plots)
is completely unchanged.

**How it works internally:**
1. Extracts all window initial states into `x0_batch` (n_windows × n) and
   input sequences into `u_batch` (n_windows × w × m).
2. Calls `jax.vmap(jax.jacfwd(simulate))` once, computing exact Jacobians
   `dY/dx₀` for every window in a single batched XLA kernel.
3. Wraps the results into the same `O_df_sliding` list format.

The first call pays a one-time JIT compilation cost (~1 s); subsequent calls
reuse the compiled kernel.

In [ ]:
w = 6  # window size in time-steps

t0 = time.perf_counter()
SEOM = JaxSlidingEmpiricalObservabilityMatrix(jax_simulator, t_sim, x_sim, u_sim, w=w)
print(f'JAX sliding observability: {time.perf_counter() - t0:.2f} s  '
      f'({len(SEOM.O_df_sliding)} windows)')

In [ ]:
O_sliding = SEOM.get_observability_matrix()
print(len(O_sliding), 'windows')

In [ ]:
# Visualize the first sliding observability matrix
OI = ObservabilityMatrixImage(O_sliding[0], cmap='bwr')
OI.plot(scale=2.0)

## Compute Fisher information matrix & inverse for first window

Identical to the original notebook — `FisherObservability` only needs
the `O_df` DataFrame, which has the same format regardless of whether it
came from the legacy or JAX class.

In [ ]:
R = {'r': 0.1}  # sensor noise variance

FO = FisherObservability(SEOM.O_df_sliding[0], R, lam=1e-8)
F, F_inv, R_mat = FO.get_fisher_information()
F_inv

## Compute Fisher information & inverse for each sliding window

Identical to the original notebook.

In [ ]:
o_sensors    = ['r']
o_states     = ['g', 'd']
o_time_steps = np.arange(0, w)

SFO = SlidingFisherObservability(SEOM.O_df_sliding, time=SEOM.t_sim, lam=1e-8, R=R,
                                 states=o_states, sensors=o_sensors,
                                 time_steps=o_time_steps, w=None)

In [ ]:
EV_aligned = SFO.get_minimum_error_variance()
EV_no_nan  = EV_aligned.bfill().ffill()

# Plot error variance as color on state time-series

Identical to the original notebook.

In [ ]:
states  = list(SFO.FO[0].O.columns)
n_state = len(states)

fig, ax = plt.subplots(n_state, 2, figsize=(6, n_state * 2), dpi=150, sharex=True)
ax = np.atleast_2d(ax)

cmap = 'inferno_r'
min_ev = np.min(EV_no_nan.iloc[:, 2:].values)
max_ev = np.max(EV_no_nan.iloc[:, 2:].values)
log_tick_high = int(np.ceil(np.log10(max_ev)))
log_tick_low  = int(np.floor(np.log10(min_ev)))
cnorm = mpl.colors.LogNorm(10**log_tick_low, 10**log_tick_high)

for n, state_name in enumerate(states):
    colorline(t_sim, x_sim[state_name], EV_no_nan[state_name].values,
              ax=ax[n, 0], cmap=cmap, norm=cnorm)
    colorline(t_sim, EV_no_nan[state_name].values, EV_no_nan[state_name].values,
              ax=ax[n, 1], cmap=cmap, norm=cnorm)

    cax  = ax[n, -1].inset_axes([1.03, 0.0, 0.04, 1.0])
    cbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=cnorm, cmap=cmap), cax=cax,
        ticks=np.logspace(log_tick_low, log_tick_high, log_tick_high - log_tick_low + 1))
    cbar.set_label('min. EV: ' + state_name, rotation=270, fontsize=7, labelpad=8)
    cbar.ax.tick_params(labelsize=6)

    x_vals = x_sim[state_name]
    ax[n, 0].set_ylim(np.min(x_vals) - 0.1, np.max(x_vals) + 0.1)
    ax[n, 0].set_ylabel('state: ' + state_name, fontsize=7)

    ax[n, 1].set_ylim(10**log_tick_low, 10**log_tick_high)
    ax[n, 1].set_yscale('log')
    ax[n, 1].set_ylabel('min. EV: ' + state_name, fontsize=7)
    ax[n, 1].set_yticks(
        np.logspace(log_tick_low, log_tick_high, log_tick_high - log_tick_low + 1))

for a in ax.flat:
    a.tick_params(axis='both', labelsize=6)
    a.set_xlabel('time (s)', fontsize=7)
    offset = t_sim[-1] * 0.05
    a.set_xlim(-offset, t_sim[-1] + offset)

fig.subplots_adjust(wspace=0.3, hspace=0.4)
plt.show()